# Reconcile ETL Cron Runs vs Dataset Tracker Metadata

This program's goal is to compare the chronological frequency of all tasks being run on the OIT server against what is listed in the Dataset Tracker, which ultimatley is teh ground truth for ALL metadata for our CIM datasets, of which the dadtaset update schedule is just a small, though very important, part.

## Library Imports

In [1]:

import pandas as pd
import plotly.express as px
import calendar
from datetime import datetime
from datetime import timedelta
from cronsim import CronSim

from datetime import date

today = date.today()
import argparse
import json
#import PySimpleGUI as sg
#import PySimpleGUIWeb as sg
import pathlib
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import textwrap
import re
import webbrowser
import subprocess
import collections
import pyglet,tkinter
from pyglet import font
import os
import subprocess
import requests
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# import OpenGL
# from OpenGL import GLU
font.add_file('/etc/fonts/fonts/CENTAUR.TTF')
font='Courier 10 bold '
import difflib

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
pd.options.mode.chained_assignment = None 
pd.options.display.max_colwidth = 100
pd.set_option('display.width', 1500)
pd.set_option('display.max_colwidth', 1000)

## Get CRON info

### Read the cron_file from the local ETL github repo

In [2]:
crons = []
def find_all(a_str, sub):
    start = 0
    while True:
        start = a_str.find(sub, start)
        if start == -1: return
        yield start
        start += len(sub) # use start += 1 to find overlapping matches
with open("/home/joe/bic_etl/general/cron/cron_file","r") as fin:
    for line in fin:
        if line[0:1] != "#"  and len(line) > 5:
           #print(line0)
#           print(line)
           crons.append(line.rstrip())
        line0=line
print(f"{len(crons)} Cron Jobs Found")

crons_all = []
ncrons=0
descriptions = []
for line in crons:
    line = line.strip(" ")
    spl = line.split(" ")
    crn=""
    print(line)
    mm = line.find("node")
    nj = line.find("java")
    npyth = line.find("python")
    nt = line.find("-t")
    np = line.find("-p")
    if (np > 0):
        a = list(find_all(line[np:],'\"'))
        # if len(a) > 0:
        #   p=line[np+a[0]:np+a[1]+1]
        # else:
        ss = line[np:].split()
        p=ss[1]
    elif mm > 0:
        pp = spl[6].split("/")
        p= pp[-1]
        
    elif nj > 0:  # java line
        pp = spl[7].split("/")
        p= pp[-1]
       
    elif npyth > 0:  # java line
        pp = spl[10].split("/")
        p= pp[-1]
       
    else:
            p=""
        
    if (nt > 0):
        ngt = line.find(">>")
        a = list(find_all(line[nt:ngt],'\"'))
        if len(a) > 0:
          t=line[nt+a[0]:nt+a[1]+1]
        else:
          ss = line[nt:].split()
          t=ss[1]
    else:
        t=""
            

    for val in spl[:5]:
        crn+= f"{val} "
    crn = crn.rstrip()
    if spl[5] == "node":
        pg = spl[6]
    else:
        pg=""
    if len(p) > 0:  #  There is a program listed
      
        pgs=pg.split("/")
       
    #    print(line)
        mo = 1
        yr=today.year
        mo=today.month
        
        
        try:
            it = CronSim(crn,datetime.strptime(f"{yr}-{mo}-01","%Y-%m-%d"))
            tmp={}
            tmp["line"]=line
            tmp["desc"] = it.explain()
            print("ME ",yr,mo,it.explain())
            descriptions.append(tmp)
            a = next(it) 
        
            while  a.month == mo:
         #       print(a.month,a.day,a.hour,a.minute,a.hour+a.minute/60)
         #       print("DAY ",calendar.day_name[a.weekday()])
          #      print(f"start:{a}   end:{a+timedelta(days=1)}")
                d = dict(Day=calendar.day_name[a.weekday()],Cron=line,T=t,TM=f"{a.hour}:{a.minute}",Task=p,Details=t,Program=pg,Start=a,End=a+timedelta(days=1),Time=a.hour+a.minute/60)
                crons_all.append(d)
            # print(f"pg: {pg} P:{p} T:{t}")    
                a = next(it)
            ncrons+=1 
        except Exception as err:
            print("Count not Process",crn)
            print(err)
            print(line)
    else:  # Not a node runnning a cim dataset... must be java or python
        print(line)
    #  print("--------")

print(f"{len(crons_all)}  Crons successfully mapped to time ranges")

30 Cron Jobs Found
50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
ME  2024 3 At 02:50 every day
58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log
ME  2024 3 At 02:58 every day
0 5 * * * PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 2>> $bic_etl_home/general/logs/cron.log
ME  2024 3 At 05:00 every day
10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log
ME  2024 3 At 03:10 every day
0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log
ME  2024 3 At 04:00 every day
5 6 * * 5 node /usr

In [13]:
print(today.today())
display(dir(today))

2024-03-10


['__add__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__ne__',
 '__new__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rsub__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 'ctime',
 'day',
 'fromisocalendar',
 'fromisoformat',
 'fromordinal',
 'fromtimestamp',
 'isocalendar',
 'isoformat',
 'isoweekday',
 'max',
 'min',
 'month',
 'replace',
 'resolution',
 'strftime',
 'timetuple',
 'today',
 'toordinal',
 'weekday',
 'year']

Sunday


In [17]:
day = datetime.now().strftime('%A')
print(day)

tod={}
for cron in crons_all:
    if cron['Day'] == day:
       print(cron['Day'],cron['Cron'])
       tod[cron['Cron']]=1

Sunday
Sunday 50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
Sunday 50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
Sunday 50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
Sunday 50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
Sunday 50 2 * * * node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 2>> $bic_etl_home/general/logs/cron.log
Sunday 58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log
Sunday 58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_

In [20]:
for key in tod:
 #   print(key)
    if key.find("node") > -1:
   #     print("Found Node")
        st = key.find("node")
        key=key[st:]
        ed = key.find("2>>")
  #      print(key[:ed])
    elif key.find("PATH") > -1:
        st = key.find("PATH")
        key=key[st:]
        ed = key.find("2>>")
  #      print(key[:ed])
    else:
        print(key)
        

58 2 * * * /usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 2>> $bic_etl_home/general/logs/cron.log


In [30]:
keys = ["node","PATH","/usr/bin"]
tod2 = tod.copy()
toRun = []
for string in tod:
 #   print(key)
    hit=0
    for key in keys:
        if string.find(key) > -1:
            st = string.find(key)
            string=string[st:]
            ed = string.find("2>>")
            hit=1
            if ed > -1:
                toRun.append(string[:ed])
            else:
                print("ROH ROH ",string)
    if hit == 0:
        print(string)
   

for string in toRun:
    print(string)

node /usr/local/cim/bic_etl/general/scripts/pull_and_setup.js 
/usr/bin/java -jar /usr/local/cim/bic_etl/general/datasync/DataSync-1.8.2.jar -t LoadPreferences -c /usr/local/cim/bic_etl/general/datasync/config.json 
PATH=$PATH:/home/giddensm/.local/bin PIPENV_PIPFILE=/usr/local/cim/bic_etl/general/metadata_updater/scripts/Pipfile pipenv run python /usr/local/cim/bic_etl/general/metadata_updater/scripts/metadata_updater.py -a 
node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 
node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 
node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit -t "Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado" -d /usr/local/cim/bic_etl/general/datasync/DataSync-1.9.1.jar 
node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/business -t "Business Entities in Colorado" 
node /usr/local/cim/bic_etl/gene

###  Get Groups and datasets listed in cron file

In [ ]:
cron4Xref = []
for ds in descriptions:
    print(ds["line"])
    st=ds["line"].find(r"-p")
    if st > 0:
       end = ds["line"][st+3:].find(r" ")
       group = ds["line"][st+2:st+3+end]
       print("Group: ",group)
       st = ds["line"].find(r"-t")
       ds["group"] = group
       if st > 0:
            st1 = ds["line"][st+3:].find('"')
            ed1 = ds["line"][st+3+st1+1:].find('"')
            dstitle = ds["line"][st+3+st1:st+3+st1+ed1+2]      
            print("title ",dstitle)
            ds["title"] = dstitle
       cron4Xref.append(ds) 
    print("-------------------------------------------------------\n")
    
            
            
      

In [ ]:
cron4Xref

<b>cron4Xref</b>  ->  a list which contains info about the different groups/datasets being updated by the CRON job.  The majority of the ETL updates are done by groups of datasets, but a few dataset specific updates are also run.  

### XREF Groups to Dataset Titles

The majority of the dataset updates in the cron file are done by group, i.e. where just a group is listed in the cron file, and the bic_etl.js program reads teh run_etl.json files for that group and gets all teh dataset titles that are needed (and all the steps).  So we need to get all the datasets listed for each group from the run_etl.json files for all the groups.  

#### Get the Unique Set of Groups

In [5]:
groups={}
for val in cron4Xref:
    group = val["group"].strip()
    print(group)
    if group not in groups:
        groups[group]=0
    groups[group]+=1

boulder
catalog
cdos/business/nonprofit
cdos/business/nonprofit
cdos/business/business
cdos/business/business
cdos/health
cdos/lobbyist
cdos/government
cdos/business/business
cdos/business/business
cdos/business/business
cdos/business/ucc
cdot/transportation_road_attributes
cdot/transportation_infrastructure
cdot/natural_resources
cdot/tops
dola/boundaries
dola/special_districts
dola/demographics
dora/regulations
cdor/revenue_marijuana
cdor/retail_reports
cdor/regulations_liquor
ceo/useia
ceo/useia


#### Get All Datasets for Each Group

In [6]:
def getTitles(string):
    titles=[]
    h = subprocess.check_output(string,shell=True)
    j = str(h).split("\\")
    for s in j:
       k = s.split(":")
       if len(k) > 1:
            title=k[1].strip()
            title=title.rstrip(",")
            titles.append(title)
    return titles
titlesG = {}
titles = {}
for key,count in groups.items():
    print(count,key)
    string = f"grep -i title /home/joe/bic_etl/{key}/run_etl.json"

    titlesG[key] = getTitles(string)
    
    for title in titlesG[key]:
      #  print(title)
        titles[title] = key
       

1 boulder
1 catalog
2 cdos/business/nonprofit
5 cdos/business/business
1 cdos/health
1 cdos/lobbyist
1 cdos/government
1 cdos/business/ucc
1 cdot/transportation_road_attributes
1 cdot/transportation_infrastructure
1 cdot/natural_resources
1 cdot/tops
1 dola/boundaries
1 dola/special_districts
1 dola/demographics
1 dora/regulations
1 cdor/revenue_marijuana
1 cdor/retail_reports
1 cdor/regulations_liquor
2 ceo/useia


In [7]:
titlesG

{'boulder': ['"Restaurant Inspections in Boulder County"',
  '"Septic Systems in Boulder County Colorado"'],
 'catalog': ['"CIM Catalog Download"'],
 'cdos/business/nonprofit': ['"Federal Tax-Exempt Subsection Codes in Colorado"',
  '"Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"',
  '"Charitable Organizations',
  '"Other State Solicitation of Charities',
  '"Charitable Purpose of the Charity in Colorado"',
  '"Paid Solicitor Solicitation Notices in Colorado"',
  '"Campaign Reports for Solicitation Notices to Charities in Colorado"',
  '"Solicitation Campaign Supervisors Listed on Solicitation Notices in Colorado"',
  '"Charity Extension Requests"',
  '"Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado"',
  '"Other Names a Registered Entity Uses to Solicit Contributions in Colorado"',
  '"Paid Solicitors Disclosed on Charity R

## Get Update Metadata from Dataset Tracker

### Get ALL Datasets in Tracker

Get ALL the datasets listed in the dataset tracker.

In [8]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=3))

    return df
            
df = getXrefs()

/tmp/ipykernel_769/2438374747.py:24: DeprecationWarning: [Deprecated][in version 6.0.0]: client_factory will be replaced by gspread.http_client types
  df = getXrefs()


In [9]:
df.columns

Index(['Dataset Title', 'Short Description', 'Category', 'Keywords', 'Type', 'License Type', 'Data Provider', 'Data Provided by', 'Source Link', 'State Steward', 'Citation', 'Agency Program Page', 'Agency Data Series Page', 'Business Contact and Phone', 'Technical Contact and Phone', 'Data Source', 'Unit of Analysis', 'Granularity Coverage', 'Geographic Extent and Division', 'Collection Mode', 'Collection Methodology', 'Data Collection Instrument', 'Date of Initial Dataset Creation', 'Field Names, comma delimited', 'Oldest Record in Dataset', 'Newest Record in Dataset', 'Long Description', 'Data Dictionary', 'Additional Metadata', 'Technical Documentation', 'Data Quality Certification', 'Applicable Information Quality Guideline Designation', 'Stewardship Plan', 'Collection Method', 'Horizontal Accuracy', 'Horizontal Coordinate System', 'Update Schedule', 'Update Method', 'Source Update Schedule', 'Update Schedule Last Audit', 'Update Type', 'Total Records at Initial Publish', 'Row Clas

In [10]:
sorted(df.columns)

['API 4x4',
 'Additional Metadata',
 'Agency Data Series Page',
 'Agency Program Page',
 'Applicable Information Quality Guideline Designation',
 'Business Contact and Phone',
 'Category',
 'Citation',
 'Collection Method',
 'Collection Methodology',
 'Collection Mode',
 'Complexity',
 'Coordinate System Disclaimer',
 'Data Collection Instrument',
 'Data Dictionary',
 'Data Provided by',
 'Data Provider',
 'Data Quality Certification',
 'Data Source',
 'Dataset Title',
 'Date Published to CIM',
 'Date of Initial Dataset Creation',
 'Expected approximate increase in record count at update',
 'FIle Size at Initial Publish or as of 3-1-2016',
 'Field Names, comma delimited',
 'Geographic Extent and Division',
 'GoCode FY Published to CIM',
 'Granularity Coverage',
 'Horizontal Accuracy',
 'Horizontal Coordinate System',
 'Keywords',
 'License Type',
 'Long Description',
 'Newest Record in Dataset',
 'Oldest Record in Dataset',
 'Quarter of Gov FY Published',
 'Related Datasets',
 'Row Cla

### Extract Just the Subset of Metadata we Need

In [11]:
#dfS = df[['Socrata Link','Dataset Title','State Steward','Category','Type','Update Type','Update Schedule', 'Update Method','Source Update Schedule','CIM Updated', 'Days Since CIM Update',
#       'cimAllData Updates', 'Complexity']]

dfS = df[['Socrata Link','Dataset Title','State Steward','Category','Type','Update Type','Update Schedule', 'Update Method','Source Update Schedule',
       'cimAllData Updates', 'Complexity']]

### Quick View of the Unique Values in Each Column

In [12]:
for col in dfS.columns:
    print("Column ",col)
    print(dfS[col].value_counts())
    print("-----------------------------------------")

Column  Socrata Link
Socrata Link
itzi-4q6v    6
8hx8-24k6    1
pxgq-s7pg    1
be63-q8yz    1
2mk-94p9     1
ifza-iedd    1
evkd-zgn4    1
e4ky-6g2n    1
5ccs-vx79    1
u943-ics6    1
5qqr-23dz    1
t8ag-kzkw    1
knbf-ggf2    1
ej2c-jkvh    1
3sm5-jtur    1
j7a3-jgd3    1
d4s4-xqg6    1
e7ye-tasg    1
gyeb-jc69    1
qvrk-xsmj    1
d6t8-xish    1
ier5-5ms2    1
kapc-ib6e    1
4zse-6bnw    1
pdd3-umrz    1
hpem-wb68    1
ab5h-juwk    1
wv7f-qjj7    1
68n7-r6rp    1
7s5z-vewr    1
349y-twqi    1
g53r-j5td    1
q5vp-adf3    1
bu8h-8sux    1
afm6-vyyp    1
3j7e-ezpr    1
sfvq-tb9q    1
mr4v-jz8u    1
2cpa-vbur    1
n55r-9hud    1
73ay-2ues    1
853a-s2qz    1
wwbh-7bpa    1
ew9y-6tv9    1
5wyf-xqw7    1
q2av-rpr5    1
3ka2-m6zm    1
mguq-rjzb    1
3thw-b7wj    1
y6w6-igw6    1
gemu-wyf3    1
q2iw-kpix    1
rpvk-ifh4    1
xz8z-8s8e    1
g62i-kdzu    1
82s5-cpkk    1
8pk9-mh2i    1
ipm7-5rxr    1
tn2f-pf3u    1
chpv-m4xq    1
hysf-mrke    1
b7zn-sry4    1
ceje-t3ek    1
24hn-nv39    1
y2yv-a

### Extract Just the Automated Dataset Updates 

The Datasets with Automated in their Update Type are the ones we need to check.  We will check for those incorrectly labled (i.e. datasets IN the cron etl process, but not listed as automatic) in different steps. 

In [13]:
dfSauto = dfS.loc[dfS["Update Type"].str.lower().str.contains("auto")]

In [14]:
dfSauto["Update Type"].value_counts()

Update Type
Automated - weekly     41
Automated - monthly    26
Automated - daily      19
Name: count, dtype: int64

In [15]:
dfCron = pd.DataFrame(cron4Xref)

In [16]:
mapCron = {
"At 04:00 on Tuesday": "Automated - weekly",
"At 04:00 every day": "Automated - daily",
"At 04:00 every day": "Automated - daily" ,
"At 03:10 every day": "Automated - daily" , 
"At 06:05 on Friday": "Automated - weekly" ,
"At 05:05 every day": "Automated - daily" ,
"At 05:10 every day": "Automated - daily",
"At 08:00 every day": "Automated - daily" ,
"At 04:15 on Tuesday": "Automated - weely" ,
"At 04:30 on Tuesday": "Automated - weekly" ,  
"At 03:00 every day": "Automated - daily",
"At 05:30 on the fourth day of every month": "Automated - monthly",
"At 04:10 on the fourth day of every month": "Automated - monthly",
"At 04:30 on the fourth day of every month": "Automated - monthly",
"At 04:50 on the fourth day of every month": "Automated - monthly",
"At 05:45 on Wednesday": "Automated - weekly",
"At 04:00 on the fourth day of every month": "Automated - monthly",
"At 05:20 on the fourth day of every month": "Automated - monthly",
"At 04:30 on the first day of every month": "Automated - monthly",
"At 03:20 every day": "Automated - daily",
"At 04:20 on the first, the eighth, the 15th, and the 22th day of month": "Automated - weekly",
"At 04:30 on the first, the eighth, the 15th, and the 22th day of month": "Automated - weekly",
"At 04:40 on the first, the eighth, the 15th, and the 22th day of month": "Automated - weekly",
"At 05:00 on Thursday": "Automated - weekly"   
}


dfCron["Update Type"] = dfCron["desc"].map(mapCron)
print(dfCron.isna().sum())

line            0
desc            0
group           0
title          18
Update Type     0
dtype: int64


In [17]:
dfCron["Update Type"].value_counts()

Update Type
Automated - weekly     11
Automated - daily       7
Automated - monthly     7
Automated - weely       1
Name: count, dtype: int64

In [18]:
dfCron.head()

,line,desc,group,title,Update Type
0,10 3 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p boulder 2>> $bic_etl_home/general/logs/cron.log,At 03:10 every day,boulder,NaN,Automated - daily
1,0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log,At 04:00 every day,catalog,NaN,Automated - daily
2,5 6 * * 5 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit 2>> $bic_etl_home/general/logs/cron.log,At 06:05 on Friday,cdos/business/nonprofit,NaN,Automated - weekly
3,"5 5 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit -t ""Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"" -d /usr/local/cim/bic_etl/general/datasync/DataSync-1.9.1.jar 2>> $bic_etl_home/general/logs/cron.log",At 05:05 every day,cdos/business/nonprofit,"""Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado""",Automated - daily
4,"10 5 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/business -t ""Business Entities in Colorado"" 2>> $bic_etl_home/general/logs/cron.log",At 05:10 every day,cdos/business/business,"""Business Entities in Colorado""",Automated - daily


In [19]:
titlesG

{'boulder': ['"Restaurant Inspections in Boulder County"',
  '"Septic Systems in Boulder County Colorado"'],
 'catalog': ['"CIM Catalog Download"'],
 'cdos/business/nonprofit': ['"Federal Tax-Exempt Subsection Codes in Colorado"',
  '"Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"',
  '"Charitable Organizations',
  '"Other State Solicitation of Charities',
  '"Charitable Purpose of the Charity in Colorado"',
  '"Paid Solicitor Solicitation Notices in Colorado"',
  '"Campaign Reports for Solicitation Notices to Charities in Colorado"',
  '"Solicitation Campaign Supervisors Listed on Solicitation Notices in Colorado"',
  '"Charity Extension Requests"',
  '"Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado"',
  '"Other Names a Registered Entity Uses to Solicit Contributions in Colorado"',
  '"Paid Solicitors Disclosed on Charity R

In [20]:
dfCronAll = pd.DataFrame({},columns=dfCron.columns)
dfCronAll.head()

,line,desc,group,title,Update Type


In [21]:
titles2Skip = dfCron["title"].value_counts().to_dict()
titles2Skip

{'"Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"': 1,
 '"Business Entities in Colorado"': 1,
 '"Business Entity Transaction History"': 1,
 '"Master List in Colorado"': 1,
 '"Trade Names for Businesses in Colorado"': 1,
 '"Trademarks for Businesses in Colorado"': 1,
 '"Gasoline Prices in Colorado"': 1,
 '"Natural Gas Prices in Colorado"': 1}

In [22]:
dfCronAll = pd.DataFrame({},columns=dfCron.columns)

for index,row in dfCron.iterrows():
    title=row['title']
    group = row["group"].strip()
    print(group)
    if group in titlesG:
        if isinstance(title,str):
            tmp=dfCron.iloc[index:index+1,:]
            print(tmp.shape)
            dfCronAll=pd.concat([tmp,dfCronAll])
        else:
            for tit in titlesG[group]:
                tit=tit.strip()
                tit=tit.replace('"','')
                print('   ',tit)
                if tit not in titles2Skip:
                    tmp=dfCron.iloc[index:index+1,:]
                    tmp["title"]=tit
                    dfCronAll=pd.concat([tmp,dfCronAll])
                else:
                    print("Skipping ",tit)
    

boulder
    Restaurant Inspections in Boulder County
    Septic Systems in Boulder County Colorado
catalog
    CIM Catalog Download
cdos/business/nonprofit
    Federal Tax-Exempt Subsection Codes in Colorado
    Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado
    Charitable Organizations
    Other State Solicitation of Charities
    Charitable Purpose of the Charity in Colorado
    Paid Solicitor Solicitation Notices in Colorado
    Campaign Reports for Solicitation Notices to Charities in Colorado
    Solicitation Campaign Supervisors Listed on Solicitation Notices in Colorado
    Charity Extension Requests
    Persons Associated with Charitable Organizations, Paid Solicitors, and Professional Fundraising Consultants in Colorado
    Other Names a Registered Entity Uses to Solicit Contributions in Colorado
    Paid Solicitors Disclosed on Charity Registration Forms in Colorado
    Charitable Solici

In [23]:
print(dfCronAll.shape)

(88, 5)


In [24]:
print(dfCronAll['title'].value_counts().sort_index())

title
"Business Entities in Colorado"                                                                                                               1
"Business Entity Transaction History"                                                                                                         1
"Gasoline Prices in Colorado"                                                                                                                 1
"Master List in Colorado"                                                                                                                     1
"Natural Gas Prices in Colorado"                                                                                                              1
"Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"    1
"Trade Names for Businesses in Colorado"                                                                                          

In [25]:
dfCronAll['group'].value_counts().sort_index()

group
 boulder                                 2
 catalog                                 1
 cdor/regulations_liquor                 7
 cdor/retail_reports                     6
 cdor/revenue_marijuana                  4
 cdos/business/business                  5
 cdos/business/nonprofit                16
 cdos/business/ucc                       4
 cdos/government                         1
 cdos/health                             1
 cdos/lobbyist                           6
 cdot/natural_resources                  2
 cdot/tops                               3
 cdot/transportation_infrastructure      4
 cdot/transportation_road_attributes     7
 ceo/useia                               2
 dola/boundaries                         2
 dola/demographics                       3
 dola/special_districts                 10
 dora/regulations                        2
Name: count, dtype: int64

In [26]:
dfCronAll.head()

,line,desc,group,title,Update Type
25,"0 5 * * 4 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p ceo/useia -t ""Natural Gas Prices in Colorado"" 2>> $bic_etl_home/general/logs/cron.log",At 05:00 on Thursday,ceo/useia,"""Natural Gas Prices in Colorado""",Automated - weekly
24,"0 4 * * 2 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p ceo/useia -t ""Gasoline Prices in Colorado"" 2>> $bic_etl_home/general/logs/cron.log",At 04:00 on Tuesday,ceo/useia,"""Gasoline Prices in Colorado""",Automated - weekly
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Manufacturer Temporary Sales Room Permits in Colorado,Automated - weekly
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Sales Rooms in Colorado,Automated - weekly
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Recently Expired and Surrendered Liquor Licenses in Colorado,Automated - weekly


### Get 4x4 for Cron Datasets From SIJ

In [27]:
def init():
    
    groups = []
    desktop = pathlib.Path("/home/joe/bic_etl")
    runEtls = []
    dataSets = []
    info = {}
    files4Datasets={}
    # .rglob() produces a generator too
    desktop.rglob("*")
    files = list(desktop.rglob("*"))
# Which you can wrap in a list() constructor to materialize
    for ff in files:
            if (str(ff).split("/")[-1] == "run_etl.json"):     
                runEtls.append(ff)
                
            
 #   print(f"{len(runEtls)} run_etl.json files found")    
    datasets = []
    dataSetsEtl={}
    dataSets = []
    groups = []
    for file in runEtls:
      f = open(file,"r")
      data = json.load(f)
   #   files[data['title']]=ff
      dataSets.append(data)
      for dat in data:
          if 'title' in dat:
                files4Datasets[dat['title']]=file
                
                
    cron4x4Titles = {}
    cronTitle4x4s = {}
    
    
    for data in dataSets:
        for ds in data:      
     #       title=ds['title']
            if 'load' in ds:
                if 'file' in ds['load']:
                    title=ds['title']
                    file=str(files4Datasets[title])
                    ne = file.rfind("/")
                    loadFile = ds['load']['file']
                    if loadFile.find('/') > -1:
                       spl=loadFile.split("/")
                       loadFile=spl[-1]

                    fileSIJ = f"{file[:ne]}/scripts/datasync/{loadFile}"
                    f = open(fileSIJ,"r")
                    sij = json.load(f)
                    cronTitle4x4s[title] = sij['datasetID'].strip()
                    cron4x4Titles[sij['datasetID'].strip()] = title
                    
    
    return cron4x4Titles,cronTitle4x4s
        

cron4x4Titles,cronTitle4x4s = init()

In [28]:
#  Remove double quotes from title names
dfCronAll["title"] = dfCronAll["title"].str.replace('"','')

In [29]:
def mapTitle(row):
    title=row['title']
    if title in cronTitle4x4s:
        w4x4= cronTitle4x4s[title]
    else:
        w4x4 = ''
    return w4x4
dfCronAll['4x4'] = dfCronAll.apply(mapTitle,axis=1)

In [30]:
dfCronAll['4x4'].value_counts()

4x4
             27
37wu-kn3g     2
8pk9-mh2i     1
9pwz-gi5v     1
d4s4-xqg6     1
htyp-tqzh     1
ier5-5ms2     1
kapc-ib6e     1
d6t8-xish     1
6kn4-89kh     1
fe4v-h3pk     1
k3gg-hhc8     1
e4ky-6g2n     1
x8tb-f3vh     1
2yhn-3dbj     1
p6y8-s74x     1
54t3-n5uh     1
v9m8-x8dh     1
j7a3-jgd3     1
7s5z-vewr     1
3sm5-jtur     1
4zse-6bnw     1
ab5h-juwk     1
wv7f-qjj7     1
q5vp-adf3     1
rkmy-yymq     1
2kvt-7ybu     1
n5ku-eixc     1
pwjb-9dd5     1
6aqe-63uq     1
ap62-sav4     1
8upq-58vz     1
wffy-3uut     1
d3m2-b6we     1
u7sb-g482     1
ej2c-jkvh     1
4am6-w6u4     1
gxnn-wthy     1
gcmk-9nwy     1
g89g-muvw     1
eqsm-7ah7     1
35k5-cv8s     1
bqa5-gr84     1
s7ct-nf65     1
k4uv-yvnk     1
casm-dbbj     1
4ykn-tg5h     1
w6kb-3vsj     1
34aw-ny67     1
wwhd-vg25     1
wwbh-7bpa     1
q2av-rpr5     1
mr4v-jz8u     1
icqv-mi3c     1
hyr8-d3v9     1
fdcw-ei67     1
ew9y-6tv9     1
7jm9-f28m     1
2z9k-uy4q     1
ihbp-hi2s     1
tuvj-xz3m     1
Name: count, dtype: 

In [32]:
dfSauto.columns

Index(['Socrata Link', 'Dataset Title', 'State Steward', 'Category', 'Type', 'Update Type', 'Update Schedule', 'Update Method', 'Source Update Schedule', 'cimAllData Updates', 'Complexity'], dtype='object')

In [33]:
dfSauto.head()

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Update Method,Source Update Schedule,cimAllData Updates,Complexity
1,s9wt-dsfz,Active Business Licenses Denver,City and county of Denver,Business,Businesses,Automated - daily,Every day after midnight,Automated on Socrata by BIC,Annually,1,
3,x5bw-ax3d,Airports in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,Automated by Staging Server,,25,
4,dm2a-biqr,All Special Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,Automated by Staging Server,,25,
11,gxnn-wthy,Bill Information and Position with Income of Lobbyist in Colorado,CDOS,Lobbyist,Bills,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,38,
20,4ykn-tg5h,Business Entities in Colorado,CDOS,Business,Businesses,Automated - daily,Every day after midnight,Automated by Staging Server,,1,


In [34]:
dfCronAll.head()

,line,desc,group,title,Update Type,4x4
25,"0 5 * * 4 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p ceo/useia -t ""Natural Gas Prices in Colorado"" 2>> $bic_etl_home/general/logs/cron.log",At 05:00 on Thursday,ceo/useia,Natural Gas Prices in Colorado,Automated - weekly,e4ky-6g2n
24,"0 4 * * 2 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p ceo/useia -t ""Gasoline Prices in Colorado"" 2>> $bic_etl_home/general/logs/cron.log",At 04:00 on Tuesday,ceo/useia,Gasoline Prices in Colorado,Automated - weekly,8pk9-mh2i
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Manufacturer Temporary Sales Room Permits in Colorado,Automated - weekly,d4s4-xqg6
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Sales Rooms in Colorado,Automated - weekly,9pwz-gi5v
23,"40 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/regulations_liquor 2>> $bic_etl_home/general/logs/cron.log","At 04:40 on the first, the eighth, the 15th, and the 22th day of month",cdor/regulations_liquor,Recently Expired and Surrendered Liquor Licenses in Colorado,Automated - weekly,pwjb-9dd5


## Analysis

### Compare Cron to Dataset Tracker

In [35]:
nbad=0
nmiss=0
ngood=0
n2many=0
miss={}
dfSauto['Cron Update'] = ""
dfSauto['Result'] = ""

for index,row in dfSauto.iterrows():
#    print(row["Update Type"],row["Dataset Title"])
    titleI=row["Dataset Title"]
    c4x4=row['Socrata Link']
    update=row["Update Type"]
    tmp = dfCronAll.loc[(dfCronAll["title"].str.lower() == titleI.lower()) | (dfCronAll['4x4'] == c4x4)]
    if tmp.shape[0] == 1:
        upt = tmp["Update Type"].values.tolist()[0]
        dfSauto.loc[index,"Con Update"]=upt

        if upt != update:
           print("BAD")
           print(titleI,upt,update,row['State Steward'])
      #     print(tmp.head())
           dfSauto.loc[index,"Result"]="BAD"
           nbad+=1
        else:
            print("GOOD ")
            print(titleI,upt,update,row['State Steward'])
      #      print(tmp.head())
            dfSauto.loc[index,"Result"]="GOOD"
                  
            ngood+=1
    elif tmp.shape[0] == 0:
        print("MISS",titleI)
        
        miss[titleI]=c4x4
      
   #    print(tmp.head())
        nmiss+=1
    else:
        print("TOO MANY",titleI)
        print(tmp.head())
        n2many+=1
        
        
print("SUMMARY")
print(f"GOOD {ngood}")
print(f"BAD {nbad}")
print(f"MISS {nmiss}")
print(f"Too Many {n2many}")

        

MISS Active Business Licenses Denver
GOOD 
Airports in Colorado Automated - monthly Automated - monthly CDOT
GOOD 
All Special Districts in Colorado Automated - monthly Automated - monthly DOLA
BAD
Bill Information and Position with Income of Lobbyist in Colorado Automated - weely Automated - weekly CDOS
GOOD 
Business Entities in Colorado Automated - daily Automated - daily CDOS
GOOD 
Business Entity Transaction History Automated - daily Automated - daily CDOS
MISS Business Improvement Districts Denver
GOOD 
Campaign Reports for Solicitation Notices to Charities in Colorado Automated - weekly Automated - weekly CDOS
GOOD 
CDOT Expenses   Automated - weekly Automated - weekly CDOT
GOOD 
CDOT Payroll Expenditures   Automated - weekly Automated - weekly CDOT
GOOD 
CDOT Revenues   Automated - weekly Automated - weekly CDOT
GOOD 
Cemetery Districts in Colorado Automated - monthly Automated - monthly DOLA
BAD
Characterization of Lobbyist Clients in Colorado Automated - weely Automated - wee

In [36]:
dfSauto.to_csv("results.csv",index=False)

In [37]:
dfSauto.head()

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Update Method,Source Update Schedule,cimAllData Updates,Complexity,Cron Update,Result,Con Update
1,s9wt-dsfz,Active Business Licenses Denver,City and county of Denver,Business,Businesses,Automated - daily,Every day after midnight,Automated on Socrata by BIC,Annually,1,,,,NaN
3,x5bw-ax3d,Airports in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,Automated by Staging Server,,25,,,GOOD,Automated - monthly
4,dm2a-biqr,All Special Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,Automated by Staging Server,,25,,,GOOD,Automated - monthly
11,gxnn-wthy,Bill Information and Position with Income of Lobbyist in Colorado,CDOS,Lobbyist,Bills,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,38,,,BAD,Automated - weely
20,4ykn-tg5h,Business Entities in Colorado,CDOS,Business,Businesses,Automated - daily,Every day after midnight,Automated by Staging Server,,1,,,GOOD,Automated - daily


###  Check Dataset Titles Not Found in Cron

In the analysis above, we start with teh titles found in the Dataset Tracker and looks for those in the cron listings.  These are the ones we did not find in the cron listing.

In [38]:
titlesDT = dfSauto["Dataset Title"].value_counts().to_dict()
titlesCron = {title.lower().strip().replace('"',''):group  for title,group in sorted(titles.items())}


In [39]:
dfSauto.columns

Index(['Socrata Link', 'Dataset Title', 'State Steward', 'Category', 'Type', 'Update Type', 'Update Schedule', 'Update Method', 'Source Update Schedule', 'cimAllData Updates', 'Complexity', 'Cron Update', 'Result', 'Con Update'], dtype='object')

In [40]:
fixed={}
# fixed['']=1
fixed['Highways in Colorado']=1
fixed['CDOT Payroll Expenditures  ']=1
fixed['Sign Locations in Colorado']=1
fixed['Sign Panels in Colorado']=1
fixed['Sign Posts in Colorado']=1
fixed['Total Revenue and Types of Art for Charities Operating in Colorado']=1
fixed['IRS Filing Information for Charities Operating in Colorado']=1
fixed['Activities of Charities Operating in Colorado']=1
fixed['Conservation Easements for Charities Operating in Colorado']=1
fixed['Purpose and Operational Size of Charities Operating in Colorado']=1
fixed['Expenses of Charities Operating in Colorado']=1
fixed['Expenses of Charities Filing IRS Form EZ Operating in Colorado']=1
fixed['Septic System Locations in Mesa County Colorado']=1
fixed['Charitable Organizations’ Offices in Colorado']=1
fixed['Other State Solicitation of Charities’ Registrants in Colorado']=1
fixed['Total Revenue of Charities Operating in Colorado']=1
fixed['Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado']=1
fixed['Fundraising Revenue of Charities Operating in Colorado']=1
fixed['Other Names a Registered Entity Uses to Solicit Contributions in Colorado']=1

In [41]:
nmiss=0
#for title in sorted(titlesDT.keys()):
for index,row in dfSauto.sort_values(by="State Steward").iterrows():
    title=row["Dataset Title"].lower().strip()
    w4x4 = row['Socrata Link']
#    print(f":{title}:")
    if title.strip() not in titlesCron and row["Dataset Title"] not in fixed:
        print(f"Length:{len(title)} Steward:{row['State Steward']}  Cat:{row['Category']}  Type:{row['Type']} Update Method: {row['Update Method']}  Days Updated:{row['cimAllData Updates']}")
        print(f"4x4: {w4x4}")
        print(f"   Title:{row['Dataset Title']}: ")
        nmiss+=1
        tmp={}
        groups=[]
        groups.append(f"{row['State Steward'].lower().strip()}/{row['Category'].lower().strip()}/{row['Type'].lower().strip()}")
        groups.append(f"{row['State Steward'].lower().strip()}/{row['Category'].lower().strip()}")
        groups.append(f"{row['State Steward'].lower().strip()}")
        
        for titleCron in titlesCron.keys():
            R=difflib.SequenceMatcher(lambda x: x == " ", a=title, b=titleCron)  
            tmp[titleCron]=R.ratio()
        tmp = {k: v for k, v in sorted(tmp.items(), key=lambda x: x[1],reverse=True)}
        ncnt=0
        for key,val in tmp.items():
            ncnt+=1
            if ncnt < 5:
                print(f"{len(key):3d} {val:.2f} {key}")
        print("-------------------")
        tmp = dfCronAll.loc[dfCronAll['group'].str.strip().isin(groups)]
        print("Titles in Cron Group: ",tmp.shape[0])
        for index,row in tmp.sort_values(by="title").iterrows():
            c4x4=row['4x4']
            titl=row['title']
            print(c4x4,titl)
        print("\n===============================================================================\n")
            
            
        
print("MISSED ",nmiss)

Length:33 Steward:City and county of Denver  Cat:Natural Resources  Type:Natural Features Update Method: Automated on Socrata by BIC  Days Updated:1
4x4: xi27-7j3e
   Title:Tree Canopy Assesment 2013 Denver: 
 47 0.33 durable medical equipment suppliers in colorado
 38 0.31 trade names for businesses in colorado
 13 0.30 cdot expenses
 13 0.30 cdot revenues
-------------------
Titles in Cron Group:  0


Length:21 Steward:City and county of Denver  Cat:Natural Resources  Type:Natural Features Update Method: Automated on Socrata by BIC  Days Updated:1
4x4: wz8h-dap6
   Title:Tree Inventory Denver: 
 19 0.45 streams in colorado
 13 0.41 cdot revenues
 18 0.41 cities in colorado
 20 0.39 counties in colorado
-------------------
Titles in Cron Group:  0


Length:31 Steward:City and county of Denver  Cat:Business  Type:Businesses Update Method: Automated on Socrata by BIC  Days Updated:1
4x4: s9wt-dsfz
   Title:Active Business Licenses Denver: 
 29 0.50 business entities in colorado
 27 0.45

In [42]:
dfS['Update Method'].value_counts()

Update Method
None                           233
Automated by Staging Server     79
Manual                          57
                                25
Automated on Socrata by BIC      7
Automated by Staging server      1
Automated by Data Provider       1
Static                           1
Name: count, dtype: int64

In [43]:
dfS['Update Type'].value_counts()

Update Type
Static                 267
Manual - annual         43
Automated - weekly      41
Automated - monthly     26
Automated - daily       19
Manual - monthly         8
Name: count, dtype: int64

In [44]:
dfS.loc[dfS['Update Type'] == "Fixed"]

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Update Method,Source Update Schedule,cimAllData Updates,Complexity


### Look at Datasets with Mis-Match in UPdate Schedule

In [45]:
dfSauto.loc[dfSauto["Result"] == "BAD"]

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Update Method,Source Update Schedule,cimAllData Updates,Complexity,Cron Update,Result,Con Update
11,gxnn-wthy,Bill Information and Position with Income of Lobbyist in Colorado,CDOS,Lobbyist,Bills,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,38,,,BAD,Automated - weely
133,g89g-muvw,Characterization of Lobbyist Clients in Colorado,CDOS,Lobbyist,Client,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,3,,,BAD,Automated - weely
191,35k5-cv8s,Directory of Lobbyist Clients in Colorado,CDOS,Lobbyist,Client,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,3,,,BAD,Automated - weely
192,bqa5-gr84,Directory of Lobbyists in Colorado,CDOS,Lobbyist,Lobbyist,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,3,,,BAD,Automated - weely
205,eqsm-7ah7,Expenses for Lobbyists in Colorado,CDOS,Lobbyist,Bills,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,3,,,BAD,Automated - weely
358,gcmk-9nwy,Subcontractors for Lobbyists in Colorado,CDOS,Lobbyist,Lobbyist,Automated - weekly,Tuesday after 12AM,Automated by Staging Server,,3,,,BAD,Automated - weely


In [46]:
dfCronAll['group'].values

array([' ceo/useia', ' ceo/useia', ' cdor/regulations_liquor',
       ' cdor/regulations_liquor', ' cdor/regulations_liquor',
       ' cdor/regulations_liquor', ' cdor/regulations_liquor',
       ' cdor/regulations_liquor', ' cdor/regulations_liquor',
       ' cdor/retail_reports', ' cdor/retail_reports',
       ' cdor/retail_reports', ' cdor/retail_reports',
       ' cdor/retail_reports', ' cdor/retail_reports',
       ' cdor/revenue_marijuana', ' cdor/revenue_marijuana',
       ' cdor/revenue_marijuana', ' cdor/revenue_marijuana',
       ' dora/regulations', ' dora/regulations', ' dola/demographics',
       ' dola/demographics', ' dola/demographics',
       ' dola/special_districts', ' dola/special_districts',
       ' dola/special_districts', ' dola/special_districts',
       ' dola/special_districts', ' dola/special_districts',
       ' dola/special_districts', ' dola/special_districts',
       ' dola/special_districts', ' dola/special_districts',
       ' dola/boundaries', ' dola/

In [47]:
a="Restaurant Inspections in Boulder Colorado"
b="Restaurant Inspections in Boulder County"


In [48]:
R=difflib.SequenceMatcher(lambda x: x == " ", a=a, b=b)

In [49]:
R.ratio()

0.8780487804878049

In [50]:
import pathlib
import json

### Title Cross Check b/w Tracker and Cron

In [51]:
dfS.head()

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Update Method,Source Update Schedule,cimAllData Updates,Complexity
0,rkuy-jxyz,2019 Novel Coronavirus COVID-19 (2019-nCoV) Data Repository by Johns Hopkins CSSE,CDOS,Health,,Static,None,None,,1439,
1,s9wt-dsfz,Active Business Licenses Denver,City and county of Denver,Business,Businesses,Automated - daily,Every day after midnight,Automated on Socrata by BIC,Annually,1,
2,vewn-5ajx,Activities of Charities Operating in Colorado,CDOS,Business,Nonprofit,Static,None,None,,1632,
3,x5bw-ax3d,Airports in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,Automated by Staging Server,,25,
4,dm2a-biqr,All Special Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,Automated by Staging Server,,25,


In [52]:
titlesTracker = df['Dataset Title'].str.lower().str.strip().values.tolist()
ngood=0
nbad=0
bad={}

for index,row in dfCronAll.iterrows():
    title = row['title']
    titleClean = title.lower().strip()
    if titleClean not in titlesTracker:
        bad[title]=row['4x4']
        nbad+=1
    else:
        ngood+=1
        
print("Good ",ngood)
print("Bad  ",nbad)

Good  82
Bad   6


In [53]:
a=df[["Socrata Link","Dataset Title"]].values.tolist()
titlesBy4x4 = {vals[0]:vals[1] for vals in a}
for title,w4x4 in bad.items():
    if w4x4 in titlesBy4x4:
        print(title,w4x4,titlesBy4x4[w4x4])
    else:
        print("MISS ",title,w4x4)
    

MISS  Retail Sales Tax Return History in Colorado 54t3-n5uh
MISS  Marijuana Sales Revenue in Colorado p6y8-s74x
Race Forecast in Colorado ab5h-juwk Race Forecasts in Colorado
MISS  Other State Solicitation of Charities 
MISS  Charitable Organizations 
MISS  CIM Catalog Download 


In [54]:
display(dfCronAll.loc[dfCronAll['title'].isin(bad)])

,line,desc,group,title,Update Type,4x4
22,"30 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/retail_reports 2>> $bic_etl_home/general/logs/cron.log","At 04:30 on the first, the eighth, the 15th, and the 22th day of month",cdor/retail_reports,Retail Sales Tax Return History in Colorado,Automated - weekly,54t3-n5uh
21,"20 4 1,8,15,22 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdor/revenue_marijuana 2>> $bic_etl_home/general/logs/cron.log","At 04:20 on the first, the eighth, the 15th, and the 22th day of month",cdor/revenue_marijuana,Marijuana Sales Revenue in Colorado,Automated - weekly,p6y8-s74x
19,30 4 1 * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p dola/demographics 2>> $bic_etl_home/general/logs/cron.log,At 04:30 on the first day of every month,dola/demographics,Race Forecast in Colorado,Automated - monthly,ab5h-juwk
2,5 6 * * 5 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit 2>> $bic_etl_home/general/logs/cron.log,At 06:05 on Friday,cdos/business/nonprofit,Other State Solicitation of Charities,Automated - weekly,
2,5 6 * * 5 node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p cdos/business/nonprofit 2>> $bic_etl_home/general/logs/cron.log,At 06:05 on Friday,cdos/business/nonprofit,Charitable Organizations,Automated - weekly,
1,0 4 * * * node /usr/local/cim/bic_etl/general/scripts/bic_etl.js -p catalog 2>> $bic_etl_home/general/logs/cron.log,At 04:00 every day,catalog,CIM Catalog Download,Automated - daily,


In [127]:
bad

{'Retail Sales Tax Return History in Colorado': '54t3-n5uh',
 'Marijuana Sales Revenue in Colorado': 'p6y8-s74x',
 'Race Forecast in Colorado': 'ab5h-juwk',
 'Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado': '',
 'Other State Solicitation of Charities': '',
 'Charitable Organizations': '',
 'CIM Catalog Download': ''}

In [131]:
titleT= df.loc[df['Socrata Link'] == '37wu-kn3g','Dataset Title'].values[0]

In [132]:
print(len(titleT))

136


In [133]:
b='Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado'

In [134]:
print(len(b))

137


In [135]:
print(titleT)
print(b)

Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado
Registration for Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado


### Clean Up Update Schedule

In [106]:
a = dfS.groupby(['Update Type','Update Schedule'],as_index=False).count()

In [109]:
b = a[['Update Type','Update Schedule','Socrata Link']]

In [111]:
b.rename(columns={'Socrata Link':'Count'},inplace=True)
display(b)

,Update Type,Update Schedule,Count
0,Automated - daily,Every day after midnight,19
1,Automated - monthly,4th of month,26
2,Automated - weekly,Tuesday after 12AM,41
3,Fixed,,1
4,Manual - Annual,Annually,6
5,Manual - annual,5-year,1
6,Manual - annual,Annually,35
7,Manual - monthly,4th of month,8
8,Static,Monthly,1
9,Static,None,261


In [ ]:
dfS.loc[dfS['Update Type'].isin(["Automated - monthly","Automated - weekly"])]

In [125]:
dfSA = dfS.loc[dfS['Update Type'].isin(["Automated - monthly","Automated - weekly"])]
dfSA['Cron Update'] = ""
for index,row in dfSA.iterrows():
    title = row['Dataset Title']
    desc = dfCronAll.loc[dfCronAll['title'].str.lower().str.strip() == title.lower().strip(),"desc"].values.tolist()
    if len(desc) > 0:
       dfSA.loc[index,'Cron Update'] = desc[0]
    


In [128]:
dfSA.columns

Index(['Socrata Link', 'Dataset Title', 'State Steward', 'Category', 'Type', 'Update Type', 'Update Schedule', 'Update Method', 'Source Update Schedule', 'cimAllData Updates', 'Complexity', 'Cron Update'], dtype='object')

In [135]:
dfSA = dfSA[['Socrata Link', 'Dataset Title', 'State Steward', 'Category', 'Type', 'Update Type', 'Update Schedule', 'Cron Update','Update Method', 'Source Update Schedule', 'cimAllData Updates'] ]

In [137]:
dfSA.sort_values(by=['Update Type','Dataset Title']).to_csv("results-updatesched.csv",index=False)

In [136]:
display(dfSA.sort_values(by=['Update Type','Dataset Title']).head(100))

,Socrata Link,Dataset Title,State Steward,Category,Type,Update Type,Update Schedule,Cron Update,Update Method,Source Update Schedule,cimAllData Updates
3,x5bw-ax3d,Airports in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,At 04:30 on the fourth day of every month,Automated by Staging Server,,24
4,dm2a-biqr,All Special Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,At 05:20 on the fourth day of every month,Automated by Staging Server,,24
29,9h8i-6khx,Cemetery Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,At 05:20 on the fourth day of every month,Automated by Staging Server,,24
139,7nuk-vzhq,Cities in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,At 04:30 on the fourth day of every month,Automated by Staging Server,,24
169,67vn-ijga,Counties in Colorado,CDOT,Transportation,Infrastructure,Automated - monthly,4th of month,At 04:30 on the fourth day of every month,Automated by Staging Server,,24
207,ua3v-vcuh,Fire Districts in Colorado,DOLA,Local Aggregation,Special Districts,Automated - monthly,4th of month,At 05:20 on the fourth day of every month,Automated by Staging Server,,24
226,9syq-9vv5,Highway Milepoints in Colorado,CDOT,Transportation,Road Attributes,Automated - monthly,4th of month,At 04:10 on the fourth day of every month,Automated by Staging Server,,24
227,trm9-dm4m,Highway Mileposts in Colorado,CDOT,Transportation,Road Attributes,Automated - monthly,4th of month,At 04:10 on the fourth day of every month,Automated by Staging Server,,24
233,xs2v-uzeg,Highway Routes in Colorado,CDOT,Transportation,Road Attributes,Automated - monthly,4th of month,At 04:10 on the fourth day of every month,Automated by Staging Server,,24
240,2h6w-z9ry,Highways in Colorado,CDOT,Transportation,Road Attributes,Automated - monthly,4th of month,At 04:10 on the fourth day of every month,Automated by Staging Server,,24
